# 03 — Findings: Visuals, Machine Learning & Conclusions
**Group P33** — Mika Ohayon, Akshaya Somasundaram, Harini Champooranan

| Notebook | Contents |
|---|---|
| `01_Data_Prep_Cleaning_Merge.ipynb` | Clean the raw BLS 2025, BLS 2022, and AIOE files, then merge them into `finalized_data.csv` |
| `02_Exploratory_Statistics.ipynb` | Descriptive statistics, correlations, model-form checks, and first look at 2022 → 2025 change |
| `03_Findings_Visuals_Models.ipynb` | Interactive visuals 1–10, machine learning models, results, and conclusions |

**Run them in order (01 → 02 → 03).** Notebooks 02 and 03 load `finalized_data.csv`, which notebook 01 creates.

### Research question
Which U.S. occupations show employment or wage changes consistent with AI displacement, and does that line up with how exposed each occupation already was to AI? Is the relationship the same for higher-skill and lower-skill occupations?

### This notebook
- **Part A — Who is exposed? (2025):** Visuals 1–7 and machine learning on wage levels
- **Part B — Is displacement showing up? (2022 → 2025):** Visuals 8–10 and machine learning on change
- **Part C — Conclusions and limitations**

All charts use **Altair** and are interactive. Colors are consistent: **blue = higher-skill**, **pink = lower-skill**. Models use **scikit-learn** and **statsmodels**.

Requires `finalized_data.csv` from notebook 01.


In [1]:
# If a package is missing, uncomment and run once:
# %pip install pandas numpy altair scikit-learn statsmodels

import warnings
import numpy as np
import pandas as pd
import altair as alt
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.inspection import permutation_importance

warnings.filterwarnings('ignore')
alt.data_transformers.disable_max_rows()
pd.set_option('display.width', 140)

In [2]:
df = pd.read_csv('finalized_data.csv', dtype={'soc_major': str, 'OCC_CODE': str})

HI, LO = 'Higher-skill (SOC 11–29)', 'Lower-skill (SOC 31–53)'
BAND_LABELS = ['Bottom 20%', '20–40%', '40–60%', '60–80%', 'Top 20%']
BAND_ORDER = BAND_LABELS[::-1]
BIN_ORDER = [f'{s:+.1f} to {s + 0.5:+.1f}' for s in sorted(df['aioe_bin_start'].unique())]

print('Loaded finalized_data.csv:', df.shape)
print(df['skill_tier'].value_counts().to_string())

Loaded finalized_data.csv: (670, 21)
skill_tier
Lower-skill (SOC 31–53)     393
Higher-skill (SOC 11–29)    277


---
# Part A — Who Is Exposed? (2025)

In [3]:
tier_scale = alt.Scale(domain=[HI, LO], range=["#0d3eb9", "#d9116bff"])   # blue = higher, orange = lower
GREY = '#d4d4d4'

def tier_color(legend=True):
    return alt.Color('skill_tier:N', scale=tier_scale,
                     legend=alt.Legend(title='Skill tier') if legend else None)

occ_tip = [
    alt.Tooltip('occupation:N', title='Occupation'),
    alt.Tooltip('major_group:N', title='Group'),
    alt.Tooltip('AIOE:Q', title='AIOE', format='.2f'),
    alt.Tooltip('A_MEDIAN:Q', title='Median wage', format='$,.0f'),
    alt.Tooltip('TOT_EMP:Q', title='Workers', format=',.0f'),
]

### Visual 1 — Who is exposed?
Average AI exposure for each occupation group (0 = the average job).
**How to use it:** click a bar to see how exposure is spread inside that group. Click empty space to reset.

In [4]:
group_stats = (df.groupby(['major_group', 'skill_tier'], as_index=False)
                 .agg(mean_aioe=('AIOE', 'mean'), workers=('TOT_EMP', 'sum'), n_occ=('AIOE', 'size')))

group_click = alt.selection_point(fields=['major_group'])

bars = (alt.Chart(group_stats).mark_bar().encode(
    y=alt.Y('major_group:N', sort='-x', title=None, axis=alt.Axis(labelLimit=220)),
    x=alt.X('mean_aioe:Q', title='Average AIOE exposure (standardized)'),
    color=alt.condition(group_click, tier_color(True), alt.value(GREY)),
    tooltip=[alt.Tooltip('major_group:N', title='Occupation group'),
             alt.Tooltip('mean_aioe:Q', title='Mean AIOE', format='.2f'),
             alt.Tooltip('n_occ:Q', title='Occupations'),
             alt.Tooltip('workers:Q', title='Workers', format=',.0f')],
).add_params(group_click).properties(width=360, height=440,
                                     title='Click a group'))

zero = pd.DataFrame({'x': [0], 'label': ['Average occupation (AIOE = 0)']})
zero_rule = alt.Chart(zero).mark_rule(color='black', strokeDash=[4, 3]).encode(x='x:Q')
zero_text = (alt.Chart(zero).mark_text(align='left', baseline='top', dx=4, dy=2, fontSize=11)
             .encode(x='x:Q', y=alt.value(0), text='label:N'))

hist = (alt.Chart(df).mark_bar().encode(
    x=alt.X('AIOE:Q', bin=alt.Bin(maxbins=25), title='AIOE of occupations in the selected group'),
    y=alt.Y('count():Q', title='Number of occupations'),
    color=tier_color(False),
    tooltip=[alt.Tooltip('count():Q', title='Occupations')],
).transform_filter(group_click).properties(width=320, height=440,
                                           title='How exposure is spread inside the selection'))

visual1 = alt.layer(bars, zero_rule, zero_text) | hist
visual1

alt.HConcatChart(...)

> **Key takeaways**
>
> - Exposure is **concentrated**, not spread evenly across the economy.
> - Most exposed groups: **Computer & Math (+1.44)**, **Legal (+1.38)**, **Business & Financial (+1.20)**.
> - Least exposed: **Construction (−1.38)** and **Building & Grounds Cleaning (−1.15)**, which are physical, hands-on jobs.
> - 86% of higher-skill jobs are above average in exposure, compared with only 20% of lower-skill jobs.

---

### Visual 2 — Does exposure pay the same in both skill tiers?
Each dot is an occupation, with AI exposure across and median wage up (log scale). Each skill tier has its own fitted line.
**How to use it:** scroll to zoom, drag to pan, click the legend to isolate a tier, and use the slider to change dot size.

In [5]:
zoom_pan = alt.selection_interval(bind='scales', encodings=['x', 'y'])      
legend_sel = alt.selection_point(fields=['skill_tier'], bind='legend')      
size_var = alt.param(value=60, bind=alt.binding_range(min=10, max=250, step=10, name='Point size: '))

points = (alt.Chart(df).mark_circle(size=size_var).encode(
    x=alt.X('AIOE:Q', title='AI Occupational Exposure (AIOE, standardized)'),
    y=alt.Y('A_MEDIAN:Q', scale=alt.Scale(type='log'), title='Median annual wage (USD, log scale)'),
    color=tier_color(True),
    opacity=alt.condition(legend_sel, alt.value(0.75), alt.value(0.08)),
    tooltip=occ_tip,
).add_params(zoom_pan, legend_sel, size_var))

fit_rows, label_rows = [], []
for tier, g in df.groupby('skill_tier'):
    slope, intercept = np.polyfit(g['AIOE'], g['log_wage'], 1)
    for x in (g['AIOE'].min(), g['AIOE'].max()):
        fit_rows.append({'skill_tier': tier, 'AIOE': x, 'A_MEDIAN': np.exp(intercept + slope * x)})
    x_end = g['AIOE'].max()
    label_rows.append({'skill_tier': tier, 'AIOE': x_end, 'A_MEDIAN': np.exp(intercept + slope * x_end),
                       'label': f'{(np.exp(slope) - 1) * 100:+.0f}% wage per +1 SD AIOE'})
fit_df, label_df = pd.DataFrame(fit_rows), pd.DataFrame(label_rows)

y_log = alt.Y('A_MEDIAN:Q', scale=alt.Scale(type='log'))
fit_lines = (alt.Chart(fit_df).mark_line(size=3, clip=True)
             .encode(x='AIOE:Q', y=y_log, color=tier_color(False)))
fit_labels = (alt.Chart(label_df).mark_text(align='right', dy=-12, fontWeight='bold', clip=True)
              .encode(x='AIOE:Q', y=y_log, text='label:N', color=tier_color(False)))


def callout(data, prefix, align, dx):
    return (alt.Chart(data).transform_calculate(label=f"'{prefix}' + datum.occupation")
            .mark_text(align=align, dx=dx, dy=-8, fontSize=11, fontStyle='italic', clip=True)
            .encode(x='AIOE:Q', y=y_log, text='label:N'))


most = callout(df.loc[[df['AIOE'].idxmax()]], 'Most exposed: ', 'right', -6)
least = callout(df.loc[[df['AIOE'].idxmin()]], 'Least exposed: ', 'left', 6)

visual2 = alt.layer(points, fit_lines, fit_labels, most, least).properties(
    width=720, height=460,
    title=alt.TitleParams('Does exposure pay the same in both skill tiers?',
                          subtitle=['Scroll to zoom · drag to pan · click the legend to isolate a tier · '
                                    'slider changes point size · lines are fitted on log wage']))
visual2

alt.LayerChart(...)

> **Key takeaways**
>
> - The lines go **up**: more exposed jobs pay more, the opposite of the usual assumption.
> - **Higher-skill:** each +1 SD of exposure goes with about **21% higher pay**.
> - **Lower-skill:** only about **4.5% higher**, a nearly flat line.
> - The two tiers behave differently, which is why our models include a **skill × exposure interaction**.

---

### Visual 3 — Who sits in the high-exposure, high-wage corner?
**How to use it:** drag a box on the left chart, for example over the upper-right corner. The same jobs light up on the right chart, and the bar below totals their workers.

In [6]:
brush = alt.selection_interval()

sc_wage = (alt.Chart(df).mark_circle(size=55).encode(
    x=alt.X('AIOE:Q', title='AIOE'),
    y=alt.Y('A_MEDIAN:Q', scale=alt.Scale(type='log'), title='Median annual wage (USD, log)'),
    color=alt.condition(brush, tier_color(True), alt.value(GREY)),
    tooltip=occ_tip,
).add_params(brush).properties(width=340, height=330, title='Drag a box here (AIOE vs wage)'))

sc_emp = (alt.Chart(df).mark_circle(size=55).encode(
    x=alt.X('AIOE:Q', title='AIOE'),
    y=alt.Y('TOT_EMP:Q', scale=alt.Scale(type='log'), title='Workers (log)'),
    color=alt.condition(brush, tier_color(False), alt.value(GREY)),
    tooltip=occ_tip,
).properties(width=340, height=330, title='The same occupations (AIOE vs employment)'))

emp_bar = (alt.Chart(df).mark_bar().encode(
    y=alt.Y('skill_tier:N', sort=[HI, LO], title=None, axis=alt.Axis(labelLimit=220)),
    x=alt.X('sum(TOT_EMP):Q', title='Workers inside the brushed region'),
    color=tier_color(False),
).transform_filter(brush))
emp_text = (emp_bar.mark_text(align='left', dx=4, fontWeight='bold')
            .encode(text=alt.Text('sum(TOT_EMP):Q', format=',.0f'), color=alt.value('black')))
emp_chart = (emp_bar + emp_text).properties(width=680, height=110,
                                            title='How many workers are in the brushed region?')

visual3 = ((sc_wage | sc_emp) & emp_chart).properties(
    title=alt.TitleParams('Who sits in the high-exposure, high-wage corner?',
                          subtitle=['Brush the upper-right of 3a: matching points light up in 3b '
                                    'and 3c totals the workers. Grey = outside the brush.']))
visual3

alt.VConcatChart(...)

> **Key takeaways**
>
> - The high-exposure, top-20%-pay corner holds about **101 occupations and 16 million workers**.
> - **95% of them are higher-skill**: general managers, management analysts, financial managers, lawyers, IT managers.
> - Exposure doesn't predict job size: brushing any region shows big and small jobs mixed together (r ≈ 0.02).

---

### Visual 4 — Which jobs are most and least exposed?
**How to use it:** use the slider to choose how many jobs to show. Click any bar to see that job's pay range (10th to 90th percentile).

In [7]:
n_param = alt.param(value=8, bind=alt.binding_range(min=3, max=15, step=1, name='Show top / bottom N: '))
occ_click = alt.selection_point(fields=['occupation'], empty=False)


def ranked(order, title):
    return (alt.Chart(df)
            .transform_window(rank='rank()', sort=[alt.SortField('AIOE', order=order)])
            .transform_filter(alt.datum.rank <= n_param)
            .mark_bar().encode(
                x=alt.X('AIOE:Q', title='AIOE'),
                y=alt.Y('occupation:N', sort='-x' if order == 'descending' else 'x',
                        title=None, axis=alt.Axis(labelLimit=260)),
                color=tier_color(order == 'descending'),
                stroke=alt.condition(occ_click, alt.value('black'), alt.value('transparent')),
                strokeWidth=alt.value(2.5),
                tooltip=occ_tip)
            .add_params(occ_click)
            .properties(width=300, height=alt.Step(22), title=title))


most_exposed = ranked('descending', 'Most exposed occupations')
least_exposed = ranked('ascending', 'Least exposed occupations')

pct_order = ['A_PCT10', 'A_PCT25', 'A_MEDIAN', 'A_PCT75', 'A_PCT90']
detail_base = (alt.Chart(df).transform_filter(occ_click)
               .transform_fold(pct_order, as_=['percentile', 'wage']))
x_pct = alt.X('percentile:N', sort=pct_order, title=None,
              axis=alt.Axis(labelAngle=0,
                            labelExpr="{'A_PCT10':'10th','A_PCT25':'25th','A_MEDIAN':'Median',"
                                      "'A_PCT75':'75th','A_PCT90':'90th'}[datum.label]"))
detail_line = detail_base.mark_line(point=True, color='#444').encode(
    x=x_pct, y=alt.Y('wage:Q', title='Annual wage (USD)'),
    tooltip=[alt.Tooltip('occupation:N', title='Occupation'),
             alt.Tooltip('percentile:N', title='Percentile'),
             alt.Tooltip('wage:Q', title='Wage', format='$,.0f')])
detail_label = (detail_base.transform_filter(alt.datum.percentile == 'A_MEDIAN')
                .mark_text(dy=-16, fontWeight='bold', fontSize=12)
                .encode(x=x_pct, y='wage:Q', text='occupation:N'))
detail = (detail_line + detail_label).properties(
    width=640, height=220, title='Click any bar above to see that job’s wage range')

visual4 = alt.vconcat(alt.hconcat(most_exposed, least_exposed), detail).add_params(n_param)
visual4

alt.VConcatChart(...)

> **Key takeaways**
>
> - Most exposed: **Genetic Counselors, Financial Examiners, Actuaries ($130k), Budget Analysts, Judges ($154k)**. These are analytical, information-heavy jobs.
> - Least exposed: **fitness trainers, construction helpers, rebar workers, garment pressers ($35–59k)**. These are physical, hands-on jobs.
> - This turns an abstract score into concrete jobs: AI exposure tracks how much of a job is reading, analyzing and deciding.

---

### Visual 5 — Is the story robust to how "high exposure" is defined?
"High exposure" depends on where we draw the line, so this chart lets you move it.
**How to use it:** drag the slider to change the exposure cutoff. The bars recompute live.

In [8]:
cut_param = alt.param(value=0.5, bind=alt.binding_range(
    min=-1.5, max=1.5, step=0.1, name='"High exposure" = AIOE above: '))
exp_scale = alt.Scale(domain=['Lower exposure', 'High exposure'], range=["#2b26ba", "#07986d"])
base5 = alt.Chart(df).transform_calculate(
    exposure=alt.expr.if_(alt.datum.AIOE > cut_param, 'High exposure', 'Lower exposure'))


def tier_exposure(y_field, title, fmt, legend):
    bars_ = base5.mark_bar().encode(
        x=alt.X('skill_tier:N', sort=[HI, LO], title=None, axis=alt.Axis(labelLimit=200, labelAngle=0)),
        xOffset=alt.XOffset('exposure:N', sort=['Lower exposure', 'High exposure']),
        y=alt.Y(y_field, title=title),
        color=alt.Color('exposure:N', scale=exp_scale,
                        legend=alt.Legend(title='Exposure group') if legend else None),
        tooltip=[alt.Tooltip('skill_tier:N', title='Skill tier'),
                 alt.Tooltip('exposure:N', title='Exposure'),
                 alt.Tooltip('count():Q', title='Occupations'),
                 alt.Tooltip(y_field, title=title, format=fmt)])
    labels_ = (bars_.mark_text(dy=-6, fontSize=11)
               .encode(text=alt.Text(y_field, format=fmt), color=alt.value('black')))
    return bars_ + labels_


wage_panel = tier_exposure('median(A_MEDIAN):Q', 'Median occupation wage (USD)', '$,.0f', True).properties(
    width=300, height=320, title='Typical wage')
emp_panel = tier_exposure('sum(TOT_EMP):Q', 'Total workers', ',.0f', False).properties(
    width=300, height=320, title='Workers in each group')

visual5 = alt.hconcat(wage_panel, emp_panel).add_params(cut_param).properties(
    title=alt.TitleParams('Is the story robust to how “high exposure” is defined?',
                          subtitle=['Drag the slider to move the exposure cutoff; bars and labels update live.']))
visual5

alt.HConcatChart(...)

> **Key takeaways**
>
> - With the cutoff at 0.5, highly exposed higher-skill jobs earn a median of **$89k vs $68k** for less exposed ones.
> - In lower-skill jobs the gap almost disappears: **$50k vs $49k**.
> - The pattern holds for every cutoff from −0.5 to 1.0, so the finding **doesn't depend on an arbitrary threshold**.

---

### Visual 6 — Where do workers actually sit?
Each cell is an exposure band × wage band, shaded by the number of workers.
**How to use it:** hover for counts. Click a cell to list its largest occupations.

In [9]:
cell_click = alt.selection_point(fields=['aioe_bin', 'wage_band'], on='click')

heat = (alt.Chart(df).mark_rect(stroke='white', strokeWidth=1).encode(
    x=alt.X('aioe_bin:O', sort=BIN_ORDER, title='AIOE exposure band (low → high)',
            axis=alt.Axis(labelAngle=-40)),
    y=alt.Y('wage_band:O', sort=BAND_ORDER, title='Median-wage band (quintiles of occupations)'),
    color=alt.Color('sum(TOT_EMP):Q', scale=alt.Scale(scheme='blues', type='sqrt'),
                    title='Workers', legend=alt.Legend(format='~s')),
    opacity=alt.condition(cell_click, alt.value(1), alt.value(0.4)),
    tooltip=[alt.Tooltip('aioe_bin:O', title='AIOE band'),
             alt.Tooltip('wage_band:O', title='Wage band'),
             alt.Tooltip('sum(TOT_EMP):Q', title='Workers', format=',.0f'),
             alt.Tooltip('count():Q', title='Occupations')],
).add_params(cell_click).properties(
    width=380, height=300,
    title=alt.TitleParams('Click a cell',
                          subtitle=['Top-right = high pay, high exposure · bottom-left = low pay, low exposure'])))

top_occ = (alt.Chart(df).transform_filter(cell_click)
           .transform_window(rank='rank()', sort=[alt.SortField('TOT_EMP', order='descending')])
           .transform_filter(alt.datum.rank <= 8)
           .mark_bar().encode(
               x=alt.X('TOT_EMP:Q', title='Workers'),
               y=alt.Y('occupation:N', sort='-x', title=None, axis=alt.Axis(labelLimit=280)),
               color=tier_color(True),
               tooltip=occ_tip)
           .properties(width=320, height=300,
                       title='Largest occupations in the clicked cell (all cells until you click)'))

visual6 = heat | top_occ
visual6

alt.HConcatChart(...)

> **Key takeaways**
>
> - About **50 million workers** are in highly exposed jobs (AIOE > 0.5).
> - About **16M** of them are in top-20% wage jobs, vs about **6M** in bottom-20% wage jobs.
> - Exposure is not the same as impact. This shows how many people *could* be affected, not who has been.

---

### Visual 7 — Does AIOE predict wages for jobs the models haven't seen?
All three models are trained on 75% of occupations and tested on the other 25%.
- **OLS with interaction:** a straight-line model where exposure can affect pay differently by skill tier. Easy to interpret.
- **Random Forest:** 300 decision trees averaged together. Can find curves and thresholds.
- **Gradient Boosting:** 300 small trees built one after another, each fixing the last one's mistakes.

**How to use it:** use the dropdown to switch models. The top-left panel shows predicted vs actual pay (closer to the dashed line = better).

In [10]:
FEATURES = ['AIOE', 'high_skill', 'AIOE_x_skill', 'log_emp']
FEATURE_LABELS = {'AIOE': 'AIOE', 'high_skill': 'Higher-skill (0/1)',
                  'AIOE_x_skill': 'AIOE × higher-skill', 'log_emp': 'log10 employment'}

X, y = df[FEATURES], df['log_wage']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

models = {
    'OLS (with interaction)': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                                   max_depth=3, random_state=42),
}

metric_rows, imp_rows, pred_frames = [], [], []
for name, mdl in models.items():
    mdl.fit(X_train, y_train)
    p_tr, p_te = mdl.predict(X_train), mdl.predict(X_test)
    for split, yt, yp in (('Train', y_train, p_tr), ('Test', y_test, p_te)):
        metric_rows.append({'model': name, 'split': split, 'r2': r2_score(yt, yp),
                            'rmse': float(np.sqrt(mean_squared_error(yt, yp)))})
    pi = permutation_importance(mdl, X_test, y_test, n_repeats=20, random_state=42, scoring='r2')
    for f, m in zip(FEATURES, pi.importances_mean):
        imp_rows.append({'model': name, 'feature': FEATURE_LABELS[f], 'importance': m})
    frame = df.loc[X_test.index, ['occupation', 'skill_tier', 'AIOE']].copy()
    frame['model'] = name
    frame['actual'] = np.exp(y_test.values)
    frame['predicted'] = np.exp(p_te)
    pred_frames.append(frame)

metrics_df, imp_df, pred_df = pd.DataFrame(metric_rows), pd.DataFrame(imp_rows), pd.concat(pred_frames)
print(metrics_df.round(3))

ols = models['OLS (with interaction)']
coef = dict(zip(FEATURES, ols.coef_))
coef_df = pd.DataFrame({'feature': [FEATURE_LABELS[f] for f in FEATURES], 'coef': ols.coef_})
pct_lo = (np.exp(coef['AIOE']) - 1) * 100
pct_hi = (np.exp(coef['AIOE'] + coef['AIOE_x_skill']) - 1) * 100

model_names = list(models)
model_select = alt.selection_point(
    fields=['model'], bind=alt.binding_select(options=model_names, name='Model: '),
    value=[{'model': model_names[0]}])

lo_, hi_ = pred_df[['actual', 'predicted']].min().min() * 0.9, pred_df[['actual', 'predicted']].max().max() * 1.1
log_scale = alt.Scale(type='log', domain=[lo_, hi_])

pts = (alt.Chart(pred_df).transform_filter(model_select).mark_circle(size=60, opacity=0.7).encode(
    x=alt.X('actual:Q', scale=log_scale, title='Actual median wage (USD, log)'),
    y=alt.Y('predicted:Q', scale=log_scale, title='Predicted median wage (USD, log)'),
    color=tier_color(True),
    tooltip=[alt.Tooltip('occupation:N', title='Occupation'),
             alt.Tooltip('AIOE:Q', format='.2f'),
             alt.Tooltip('actual:Q', title='Actual', format='$,.0f'),
             alt.Tooltip('predicted:Q', title='Predicted', format='$,.0f')],
).add_params(model_select))
diag_df = pd.DataFrame({'x': [lo_, hi_], 'y': [lo_, hi_]})
diag = alt.Chart(diag_df).mark_line(strokeDash=[5, 4], color='black').encode(
    x=alt.X('x:Q', scale=log_scale), y=alt.Y('y:Q', scale=log_scale))
diag_lbl = (alt.Chart(pd.DataFrame({'x': [hi_ * 0.8], 'y': [hi_ * 0.8], 'label': ['Perfect prediction (y = x)']}))
            .mark_text(align='right', dy=-10, fontSize=11)
            .encode(x=alt.X('x:Q', scale=log_scale), y=alt.Y('y:Q', scale=log_scale), text='label:N'))
scatter7 = alt.layer(pts, diag, diag_lbl).properties(width=360, height=320,
                                                     title='Held-out occupations: actual vs predicted')

importance = (alt.Chart(imp_df).transform_filter(model_select).mark_bar(color="#0576cc").encode(
    x=alt.X('importance:Q', title='Permutation importance (drop in test R²)'),
    y=alt.Y('feature:N', sort='-x', title=None, axis=alt.Axis(labelLimit=200)),
    tooltip=[alt.Tooltip('feature:N', title='Feature'),
             alt.Tooltip('importance:Q', title='Importance', format='.3f')],
).properties(width=330, height=320, title='Which inputs matter for the selected model?'))

r2_bars = (alt.Chart(metrics_df).mark_bar().encode(
    x=alt.X('model:N', title=None, axis=alt.Axis(labelAngle=-15, labelLimit=150)),
    xOffset='split:N',
    y=alt.Y('r2:Q', title='R² (on log wage)'),
    color=alt.Color('split:N', scale=alt.Scale(domain=['Train', 'Test'], range=["#baff9a", "#088c21"]),
                    title='Data split'),
    opacity=alt.condition(model_select, alt.value(1), alt.value(0.35)),
    tooltip=[alt.Tooltip('model:N', title='Model'), alt.Tooltip('split:N', title='Split'),
             alt.Tooltip('r2:Q', title='R²', format='.3f'), alt.Tooltip('rmse:Q', title='RMSE (log)', format='.3f')]))
r2_text = (r2_bars.mark_text(dy=-6, fontSize=10)
           .encode(text=alt.Text('r2:Q', format='.2f'), color=alt.value('black'), opacity=alt.value(1)))
r2_chart = (r2_bars + r2_text).properties(width=360, height=260,
                                          title='Accuracy: train vs test (gap = overfitting)')

coef_bars = (alt.Chart(coef_df).mark_bar().encode(
    y=alt.Y('feature:N', sort=None, title=None, axis=alt.Axis(labelLimit=200)),
    x=alt.X('coef:Q', title='OLS coefficient (log-wage points)'),
    color=alt.condition(alt.datum.coef > 0, alt.value("#ffe643"), alt.value("#ff4b37")),
    tooltip=[alt.Tooltip('feature:N', title='Feature'), alt.Tooltip('coef:Q', title='Coefficient', format='.3f')]))
coef_zero = alt.Chart(pd.DataFrame({'x': [0]})).mark_rule(color='black').encode(x='x:Q')
coef_chart = (coef_bars + coef_zero).properties(
    width=330, height=260,
    title=alt.TitleParams('OLS with interaction term',
                          subtitle=[f'+1 SD AIOE → {pct_lo:+.1f}% wage (lower-skill), {pct_hi:+.1f}% (higher-skill)']))

row1 = alt.hconcat(scatter7, importance).resolve_scale(color='independent')
row2 = alt.hconcat(r2_chart, coef_chart).resolve_scale(color='independent')
visual7 = alt.vconcat(row1, row2).resolve_scale(color='independent').properties(
    title=alt.TitleParams('Does AIOE actually predict wages out of sample?',
                          subtitle=['Use the dropdown to switch models; 7a–7b update, 7c highlights the chosen model.']))
visual7

                    model  split     r2   rmse
0  OLS (with interaction)  Train  0.402  0.311
1  OLS (with interaction)   Test  0.485  0.302
2           Random Forest  Train  0.718  0.214
3           Random Forest   Test  0.460  0.309
4       Gradient Boosting  Train  0.768  0.194
5       Gradient Boosting   Test  0.443  0.314


alt.VConcatChart(...)

> **Key takeaways**
>
> - **OLS: train 0.40, test 0.49.** No overfitting. It explains about half of the pay differences between jobs it hadn't seen.
> - **Random Forest (0.72 → 0.46)** and **Gradient Boosting (0.77 → 0.44)** score much higher on training data than test data, so they **overfit**.
> - The complex models don't beat the simple one, so the relationship is basically linear and **OLS is the model to trust**.
> - Skill tier is the most important input. Exposure matters too, mostly through its interaction with skill.

---

## Part 5 — Machine Learning on 2025 Wage Levels (detail)
The same three models as Visual 7, now with **5-fold cross-validation**, which is more reliable than one 75/25 split. We also fit OLS with robust standard errors to get p-values.

**Why these models:** OLS with an interaction term tests the hypothesis directly and is easy to interpret. Random Forest and Gradient Boosting check whether a flexible model finds patterns the linear one misses. The gap between train and test scores shows overfitting.

In [11]:
def make_models():
    return {
        'OLS (with interaction)': LinearRegression(),
        'Random Forest': RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=42),
        'Gradient Boosting': GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                                       max_depth=3, random_state=42),
    }

KF = KFold(n_splits=5, shuffle=True, random_state=42)


def evaluate(data, features, target, labels):
    # Train all three models on one target. Returns metrics, permutation importance, fitted models.
    X, y = data[features], data[target]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42)
    metric_rows, imp_rows = [], []
    for name, mdl in make_models().items():
        mdl.fit(X_tr, y_tr)
        cv = cross_val_score(make_models()[name], X, y, cv=KF, scoring='r2')
        metric_rows.append({
            'target': target, 'model': name,
            'train_r2': r2_score(y_tr, mdl.predict(X_tr)),
            'test_r2': r2_score(y_te, mdl.predict(X_te)),
            'cv_r2_mean': cv.mean(), 'cv_r2_sd': cv.std(),
            'test_rmse': float(np.sqrt(mean_squared_error(y_te, mdl.predict(X_te)))),
        })
        pi = permutation_importance(mdl, X_te, y_te, n_repeats=20, random_state=42, scoring='r2')
        for f, m in zip(features, pi.importances_mean):
            imp_rows.append({'target': target, 'model': name, 'feature': labels[f], 'importance': m})
    return pd.DataFrame(metric_rows), pd.DataFrame(imp_rows)


LEVEL_FEATURES = ['AIOE', 'high_skill', 'AIOE_x_skill', 'log_emp']
LEVEL_LABELS = {'AIOE': 'AIOE', 'high_skill': 'Higher-skill (0/1)',
                'AIOE_x_skill': 'AIOE × higher-skill', 'log_emp': 'log10 employment'}

level_metrics, level_imp = evaluate(df, LEVEL_FEATURES, 'log_wage', LEVEL_LABELS)
print('Original models — target: log median wage (2025)\n')
print(level_metrics.drop(columns='target').round(3).to_string(index=False))

ols_level = smf.ols('log_wage ~ AIOE * high_skill + log_emp', data=df).fit(cov_type='HC1')
print('\nOLS with robust standard errors (full sample):')
print(ols_level.summary().tables[1])
b = ols_level.params
print(f"\n+1 SD AIOE → {100*(np.exp(b['AIOE'])-1):+.1f}% wage (lower-skill), "
      f"{100*(np.exp(b['AIOE']+b['AIOE:high_skill'])-1):+.1f}% (higher-skill)")

Original models — target: log median wage (2025)

                 model  train_r2  test_r2  cv_r2_mean  cv_r2_sd  test_rmse
OLS (with interaction)     0.402    0.485       0.417     0.069      0.302
         Random Forest     0.718    0.460       0.387     0.084      0.309
     Gradient Boosting     0.768    0.443       0.379     0.088      0.314

OLS with robust standard errors (full sample):
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept          11.1645      0.083    134.052      0.000      11.001      11.328
AIOE                0.0527      0.017      3.112      0.002       0.019       0.086
high_skill          0.3276      0.044      7.425      0.000       0.241       0.414
AIOE:high_skill     0.1375      0.040      3.457      0.001       0.060       0.215
log_emp            -0.0604      0.017     -3.596      0.000      -0.093      -0.027

+1 SD AIOE → 

> **Key takeaways**
>
> - **Cross-validation confirms Visual 7:** OLS averages R² = 0.42 across 5 folds and beats Random Forest (0.39) and Gradient Boosting (0.38).
> - **AIOE (p = 0.002):** in lower-skill jobs, +1 SD of exposure goes with about **+5% pay**.
> - **AIOE × higher-skill (p = 0.001):** in higher-skill jobs the effect is about **+21%**. The difference between tiers is statistically significant.
> - **Higher-skill (p < 0.001):** higher-skill jobs pay about 39% more at average exposure.
> - **Limitation:** this is one year of data, so it shows *who is exposed*, not *who is being displaced*. Part 6 tests change.

---
# Part B — Is Displacement Showing Up? (2022 → 2025)
Part A describes **levels** in one year. To test displacement we compare **May 2022** (just before ChatGPT launched) with **May 2025**. Wages rose everywhere because of inflation, so what matters is whether exposed jobs grew **more or less than other jobs**.

In [12]:
MIN_EMP = 1000
chg = df.dropna(subset=['A_MEDIAN_22', 'TOT_EMP_22']).copy()
chg = chg[(chg['TOT_EMP_22'] >= MIN_EMP) & (chg['TOT_EMP'] >= MIN_EMP)].copy()

chg['wage_growth'] = np.log(chg['A_MEDIAN'] / chg['A_MEDIAN_22'])
chg['emp_growth'] = np.log(chg['TOT_EMP'] / chg['TOT_EMP_22'])
chg['wage_growth_pct'] = 100 * (np.exp(chg['wage_growth']) - 1)
chg['emp_growth_pct'] = 100 * (np.exp(chg['emp_growth']) - 1)
chg['rel_wage_growth'] = 100 * (chg['wage_growth'] - chg['wage_growth'].median())
chg['rel_emp_growth'] = 100 * (chg['emp_growth'] - chg['emp_growth'].median())
chg['log_wage_22'] = np.log(chg['A_MEDIAN_22'])
chg['log_emp_22'] = np.log10(chg['TOT_EMP_22'])
chg['aioe_quintile'] = pd.qcut(chg['AIOE'], 5, labels=['Q1 (least)', 'Q2', 'Q3', 'Q4', 'Q5 (most)']).astype(str)

print(f'Change-analysis sample: {len(chg)} occupations')
print(f"Typical (median) occupation: wages {chg['wage_growth_pct'].median():+.1f}%, "
      f"employment {chg['emp_growth_pct'].median():+.1f}% from 2022 to 2025")
print('\nCorrelation of AIOE with wage growth:       r =', round(chg['AIOE'].corr(chg['wage_growth']), 3))
print('Correlation of AIOE with employment growth: r =', round(chg['AIOE'].corr(chg['emp_growth']), 3))

Change-analysis sample: 661 occupations
Typical (median) occupation: wages +13.1%, employment +2.9% from 2022 to 2025

Correlation of AIOE with wage growth:       r = -0.106
Correlation of AIOE with employment growth: r = 0.119


> **Key takeaways**
>
> - **661 occupations** are compared. The typical job's pay rose **13.1%** and employment rose **2.9%**.
> - Full descriptive results are in notebook 02. Here we visualize and model them.

---

### Visual 8 — Did more-exposed jobs grow differently?
Each dot is an occupation. The y-axis is growth **relative to the typical occupation** (0 = grew like everyone else). Lines are fitted separately for each skill tier, and the label gives the slope: extra growth per +1 SD of AI exposure.

*Interactivity:* dropdown switches wage vs. employment growth, click the legend to isolate a tier, scroll to zoom, drag to pan, hover for details.

In [13]:
tier_scale = alt.Scale(domain=[HI, LO], range=['#0d3eb9', '#d9116b'])
METRICS = {'rel_wage_growth': 'Wage growth', 'rel_emp_growth': 'Employment growth'}

long = chg.melt(id_vars=['occupation', 'skill_tier', 'major_group', 'AIOE', 'wage_growth_pct', 'emp_growth_pct',
                         'A_MEDIAN_22', 'A_MEDIAN', 'TOT_EMP_22', 'TOT_EMP'],
                value_vars=list(METRICS), var_name='metric_key', value_name='rel_growth')
long['metric'] = long['metric_key'].map(METRICS)

fit_rows, lbl_rows = [], []
for (mk, tier), g in long.groupby(['metric_key', 'skill_tier']):
    slope, icpt = np.polyfit(g['AIOE'], g['rel_growth'], 1)
    xs = (g['AIOE'].min(), g['AIOE'].max())
    for x in xs:
        fit_rows.append({'metric': METRICS[mk], 'skill_tier': tier, 'AIOE': x, 'rel_growth': icpt + slope * x})
    lbl_rows.append({'metric': METRICS[mk], 'skill_tier': tier, 'AIOE': xs[1], 'rel_growth': icpt + slope * xs[1],
                     'label': f'{slope:+.1f} pts per +1 SD AIOE'})
fit_df, lbl_df = pd.DataFrame(fit_rows), pd.DataFrame(lbl_rows)

metric_sel = alt.selection_point(fields=['metric'], value=[{'metric': 'Employment growth'}],
                                 bind=alt.binding_select(options=['Employment growth', 'Wage growth'], name='Show: '))
tier_sel = alt.selection_point(fields=['skill_tier'], bind='legend')
zoom = alt.selection_interval(bind='scales')

lo_y, hi_y = long['rel_growth'].quantile([0.01, 0.99])
y_enc = alt.Y('rel_growth:Q', title='Growth vs typical occupation (percentage points)',
              scale=alt.Scale(domain=[float(np.floor(lo_y)), float(np.ceil(hi_y))]))
pts8 = (alt.Chart(long).mark_circle(size=55, clip=True).encode(
    x=alt.X('AIOE:Q', title='AI Occupational Exposure (AIOE, standardized)'),
    y=y_enc,
    color=alt.Color('skill_tier:N', scale=tier_scale, title='Skill tier'),
    opacity=alt.condition(tier_sel, alt.value(0.7), alt.value(0.06)),
    tooltip=[alt.Tooltip('occupation:N', title='Occupation'),
             alt.Tooltip('major_group:N', title='Group'),
             alt.Tooltip('AIOE:Q', format='.2f'),
             alt.Tooltip('A_MEDIAN_22:Q', title='Median wage 2022', format='$,.0f'),
             alt.Tooltip('A_MEDIAN:Q', title='Median wage 2025', format='$,.0f'),
             alt.Tooltip('wage_growth_pct:Q', title='Wage growth %', format='+.1f'),
             alt.Tooltip('TOT_EMP_22:Q', title='Workers 2022', format=',.0f'),
             alt.Tooltip('TOT_EMP:Q', title='Workers 2025', format=',.0f'),
             alt.Tooltip('emp_growth_pct:Q', title='Employment growth %', format='+.1f')],
).add_params(metric_sel, tier_sel, zoom).transform_filter(metric_sel))

zero = alt.Chart(pd.DataFrame({'y': [0]})).mark_rule(strokeDash=[5, 4], color='black').encode(y='y:Q')
zero_lbl = alt.Chart(pd.DataFrame({'y': [0], 'x': [-2.6], 't': ['Typical occupation']})).mark_text(
    align='left', dy=8, fontSize=11, fontStyle='italic').encode(x='x:Q', y='y:Q', text='t:N')
lines8 = alt.Chart(fit_df).mark_line(size=3, clip=True).encode(
    x='AIOE:Q', y='rel_growth:Q', color=alt.Color('skill_tier:N', scale=tier_scale, legend=None)
).transform_filter(metric_sel)
lbl_df['label'] = lbl_df['skill_tier'].str.split(' ').str[0] + ': ' + lbl_df['label']
lbl_df['ypix'] = np.where(lbl_df['skill_tier'] == HI, 16, 34)
lbls8 = alt.Chart(lbl_df).mark_text(align='left', x=10, fontWeight='bold', fontSize=12).encode(
    y=alt.Y('ypix:Q', scale=None), text='label:N',
    color=alt.Color('skill_tier:N', scale=tier_scale, legend=None)
).transform_filter(metric_sel)

visual8 = alt.layer(pts8, zero, zero_lbl, lines8, lbls8).properties(
    width=720, height=440,
    title=alt.TitleParams('Visual 8 — Did more-exposed jobs grow differently, 2022 → 2025?',
                          subtitle=['Dropdown switches wage vs employment · click legend to isolate a tier · '
                                    'scroll to zoom (a few extreme outliers are off-screen) · 0 = grew like the typical occupation']))
visual8

alt.LayerChart(...)

> **Key takeaways**
>
> - **Employment (default view):** more exposure goes with **faster growth in higher-skill jobs (+1.4 pts per SD)** and **slower growth in lower-skill jobs (−1.9 pts per SD)**. The lines point in opposite directions.
> - **Wages (switch with the dropdown):** both lines are basically flat (−0.9 and 0.0 pts per SD). AI exposure **did not affect raises**.
> - Dots are widely scattered, so exposure explains only a small part of why any one job grew or shrank.

---

### Visual 9 — Growth by exposure level
Occupations are split into five equal-size groups by AIOE (Q1 = least exposed, Q5 = most exposed). Bars show the **median** growth in each group, separately for each skill tier. If AI were already displacing exposed workers, the Q5 bars would be clearly lower than the Q1 bars.

*Interactivity:* dropdown switches wage vs employment, hover shows the number of occupations and workers behind each bar.

In [14]:
q_long = (chg.melt(id_vars=['skill_tier', 'aioe_quintile', 'TOT_EMP'],
                   value_vars=['wage_growth_pct', 'emp_growth_pct'], var_name='mk', value_name='growth'))
q_long['metric'] = q_long['mk'].map({'wage_growth_pct': 'Wage growth', 'emp_growth_pct': 'Employment growth'})
q_tbl = (q_long.groupby(['metric', 'skill_tier', 'aioe_quintile'], as_index=False)
               .agg(median_growth=('growth', 'median'), n=('growth', 'size'), workers=('TOT_EMP', 'sum')))
overall = (q_long.groupby('metric', as_index=False)['growth'].median()
                 .rename(columns={'growth': 'overall'}))

metric_sel9 = alt.selection_point(fields=['metric'], value=[{'metric': 'Employment growth'}],
                                  bind=alt.binding_select(options=['Employment growth', 'Wage growth'], name='Show: '))
Q_ORDER = ['Q1 (least)', 'Q2', 'Q3', 'Q4', 'Q5 (most)']

bars9 = alt.Chart(q_tbl).mark_bar().encode(
    x=alt.X('aioe_quintile:N', sort=Q_ORDER, title='AIOE quintile (least → most exposed)', axis=alt.Axis(labelAngle=0)),
    xOffset=alt.XOffset('skill_tier:N', sort=[HI, LO]),
    y=alt.Y('median_growth:Q', title='Median growth 2022 → 2025 (%)'),
    color=alt.Color('skill_tier:N', scale=tier_scale, title='Skill tier'),
    tooltip=[alt.Tooltip('skill_tier:N', title='Tier'), alt.Tooltip('aioe_quintile:N', title='Exposure group'),
             alt.Tooltip('median_growth:Q', title='Median growth %', format='+.1f'),
             alt.Tooltip('n:Q', title='Occupations'), alt.Tooltip('workers:Q', title='Workers 2025', format=',.0f')],
).add_params(metric_sel9).transform_filter(metric_sel9)
vals9 = bars9.mark_text(dy=-7, fontSize=10).encode(text=alt.Text('median_growth:Q', format='+.1f'),
                                                   color=alt.value('black'))
ref9 = alt.Chart(overall).mark_rule(strokeDash=[5, 4], color='black').encode(y='overall:Q').transform_filter(metric_sel9)
ref9_lbl = alt.Chart(overall).mark_text(align='left', x=5, dy=-8, fontSize=11, fontStyle='italic').encode(
    y='overall:Q', text=alt.value('All-occupation median')).transform_filter(metric_sel9)

visual9 = alt.layer(bars9, vals9, ref9, ref9_lbl).properties(
    width=680, height=380,
    title=alt.TitleParams('Visual 9 — Median growth by exposure group and skill tier',
                          subtitle=['Dropdown switches metric · dashed line = median across all occupations · '
                                    'hover for counts']))
visual9

alt.LayerChart(...)

> **Key takeaways**
>
> - The lower-skill jobs that shrank are mostly **Office & Admin Support** and **Sales**: telemarketers, order clerks, loan interviewers, switchboard operators.
> - These are routine, information-processing jobs that AI and automation tools can take over, which fits an early displacement story.
> - Higher-skill exposed jobs (managers, analysts, tech) **grew faster than average**. That looks more like AI *helping* these workers than replacing them.

---

### Machine learning on change (2022 → 2025)
Same three models, same 75/25 split and random seed, plus 5-fold cross-validation. Two targets: **wage growth** and **employment growth** (log changes).

Inputs: AIOE, higher-skill (0/1), the AIOE × higher-skill interaction, and two starting-point controls — the 2022 wage and 2022 employment. The controls matter because low-paid jobs tended to get bigger raises after 2022, and small occupations swing more; without them, AIOE could pick up those effects instead of anything AI-related.

**Reading the OLS output:** the AIOE coefficient × 100 ≈ extra percentage points of growth per +1 SD of exposure for **lower-skill** jobs; AIOE + interaction is the same for **higher-skill** jobs.

**What R² to expect:** change is much harder to predict than levels, so R² will be low. A low R² here is itself a finding: it means exposure doesn't explain much of who grew or shrank.

In [15]:
CHG_FEATURES = ['AIOE', 'high_skill', 'AIOE_x_skill', 'log_wage_22', 'log_emp_22']
CHG_LABELS = {'AIOE': 'AIOE', 'high_skill': 'Higher-skill (0/1)', 'AIOE_x_skill': 'AIOE × higher-skill',
              'log_wage_22': 'log wage 2022', 'log_emp_22': 'log10 employment 2022'}
TARGET_NAMES = {'wage_growth': 'Wage growth', 'emp_growth': 'Employment growth'}

chg_metrics, chg_imp, ols_change = [], [], {}
for t in TARGET_NAMES:
    m, imp = evaluate(chg, CHG_FEATURES, t, CHG_LABELS)
    chg_metrics.append(m); chg_imp.append(imp)
    ols_change[t] = smf.ols(f'{t} ~ AIOE * high_skill + log_wage_22 + log_emp_22', data=chg).fit(cov_type='HC1')
chg_metrics, chg_imp = pd.concat(chg_metrics), pd.concat(chg_imp)
chg_metrics['target'] = chg_metrics['target'].map(TARGET_NAMES)
chg_imp['target'] = chg_imp['target'].map(TARGET_NAMES)

print('Model performance on CHANGE targets\n')
print(chg_metrics.round(3).to_string(index=False))
for t, res in ols_change.items():
    print(f'\n===== OLS: {TARGET_NAMES[t]} (robust SE) =====')
    print(res.summary().tables[1])

Model performance on CHANGE targets

           target                  model  train_r2  test_r2  cv_r2_mean  cv_r2_sd  test_rmse
      Wage growth OLS (with interaction)     0.083   -0.101      -0.039     0.091      0.060
      Wage growth          Random Forest     0.645    0.081       0.084     0.050      0.055
      Wage growth      Gradient Boosting     0.772   -0.033      -0.006     0.103      0.059
Employment growth OLS (with interaction)     0.106    0.092       0.055     0.057      0.137
Employment growth          Random Forest     0.631    0.054      -0.001     0.068      0.140
Employment growth      Gradient Boosting     0.741   -0.165      -0.065     0.101      0.155

===== OLS: Wage growth (robust SE) =====
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept           0.5032      0.113      4.442      0.000       0.281       0.725
AIOE              

> **Key takeaways**
>
> - **Wage growth:** no exposure effect in either tier (p = 0.62 and 0.84). The only significant input is the **2022 wage**: lower-paid jobs got bigger raises, a post-COVID labor-market effect, not an AI effect.
> - **Employment growth:** in **lower-skill jobs, +1 SD of exposure → about 2.8 pts slower growth (p = 0.02)**, which is statistically significant.
> - In higher-skill jobs the effect is slightly positive. The gap between tiers is **not yet significant (p = 0.11)**.
> - **Prediction is weak:** test and CV R² are near zero (best about 0.08). The tree models score 0.6–0.8 on training data, so they are memorizing noise.

---

### Visual 10 — Do the models find an AI effect on change?
Left: train, test, and 5-fold CV R² for each model. Right: OLS coefficients with 95% confidence intervals — a bar whose interval crosses zero is **not** statistically distinguishable from no effect. Bottom: which inputs each model relies on (permutation importance).

*Interactivity:* dropdown switches the target; a second dropdown picks the model for the importance panel; hover for exact values.

In [16]:
coef_rows = []
for t, res in ols_change.items():
    ci = res.conf_int()
    for term in ['AIOE', 'high_skill', 'AIOE:high_skill', 'log_wage_22', 'log_emp_22']:
        coef_rows.append({'target': TARGET_NAMES[t], 'term': {'AIOE:high_skill': 'AIOE × higher-skill',
                                                              'high_skill': 'Higher-skill (0/1)',
                                                              'log_wage_22': 'log wage 2022',
                                                              'log_emp_22': 'log10 employment 2022'}.get(term, term),
                          'coef': 100 * res.params[term], 'lo': 100 * ci.loc[term, 0], 'hi': 100 * ci.loc[term, 1],
                          'p': res.pvalues[term]})
coef_df = pd.DataFrame(coef_rows)
coef_df['significant'] = np.where(coef_df['p'] < 0.05, 'p < 0.05', 'not significant')

r2_long = chg_metrics.melt(id_vars=['target', 'model'], value_vars=['train_r2', 'test_r2', 'cv_r2_mean'],
                           var_name='split', value_name='r2')
r2_long['split'] = r2_long['split'].map({'train_r2': 'Train', 'test_r2': 'Test', 'cv_r2_mean': '5-fold CV'})

target_sel = alt.selection_point(fields=['target'], value=[{'target': 'Wage growth'}],
                                 bind=alt.binding_select(options=list(TARGET_NAMES.values()), name='Target: '))
model_sel = alt.selection_point(fields=['model'], value=[{'model': 'OLS (with interaction)'}],
                                bind=alt.binding_select(options=list(make_models()), name='Model (importance): '))

r2_bars = alt.Chart(r2_long).mark_bar().encode(
    x=alt.X('model:N', title=None, axis=alt.Axis(labelAngle=-15, labelLimit=160)),
    xOffset=alt.XOffset('split:N', sort=['Train', 'Test', '5-fold CV']),
    y=alt.Y('r2:Q', title='R²'),
    color=alt.Color('split:N', sort=['Train', 'Test', '5-fold CV'],
                    scale=alt.Scale(range=['#baff9a', '#088c21', '#045215']), title='Data split'),
    tooltip=['model:N', 'split:N', alt.Tooltip('r2:Q', format='.3f')],
).add_params(target_sel).transform_filter(target_sel)
r2_txt = r2_bars.mark_text(dy=-6, fontSize=9).encode(text=alt.Text('r2:Q', format='.2f'), color=alt.value('black'))
r2_chart = (r2_bars + r2_txt).properties(width=340, height=260, title='Accuracy: train vs test vs CV')

base_c = alt.Chart(coef_df).transform_filter(target_sel)
ci_rule = base_c.mark_rule(size=2).encode(
    y=alt.Y('term:N', sort=None, title=None, axis=alt.Axis(labelLimit=200)),
    x=alt.X('lo:Q', title='OLS coefficient (percentage points of growth)'), x2='hi:Q',
    color=alt.Color('significant:N', scale=alt.Scale(domain=['p < 0.05', 'not significant'],
                                                     range=['#ff4b37', '#9e9e9e']), title='Significance'))
ci_pt = base_c.mark_point(filled=True, size=90).encode(
    y=alt.Y('term:N', sort=None), x='coef:Q', color=alt.Color('significant:N', legend=None,
                                                              scale=alt.Scale(domain=['p < 0.05', 'not significant'],
                                                                              range=['#ff4b37', '#9e9e9e'])),
    tooltip=['term:N', alt.Tooltip('coef:Q', title='Coef (pts)', format='+.2f'),
             alt.Tooltip('lo:Q', title='95% CI low', format='+.2f'), alt.Tooltip('hi:Q', title='95% CI high', format='+.2f'),
             alt.Tooltip('p:Q', title='p-value', format='.3f')])
zero10 = alt.Chart(pd.DataFrame({'x': [0]})).mark_rule(strokeDash=[4, 4], color='black').encode(x='x:Q')
coef_chart = alt.layer(ci_rule, ci_pt, zero10).properties(
    width=340, height=260, title=alt.TitleParams('OLS coefficients with 95% CI',
                                                 subtitle=['Interval crossing 0 = no clear effect']))

imp_chart = alt.Chart(chg_imp).mark_bar(color='#0576cc').encode(
    x=alt.X('importance:Q', title='Permutation importance (drop in test R²)'),
    y=alt.Y('feature:N', sort='-x', title=None, axis=alt.Axis(labelLimit=200)),
    tooltip=['feature:N', alt.Tooltip('importance:Q', format='.3f')],
).add_params(model_sel).transform_filter(target_sel).transform_filter(model_sel).properties(
    width=720, height=170, title='Which inputs matter for the selected model?')

visual10 = alt.vconcat(alt.hconcat(r2_chart, coef_chart).resolve_scale(color='independent'),
                       imp_chart).resolve_scale(color='independent').properties(
    title=alt.TitleParams('Visual 10 — Does AI exposure predict 2022 → 2025 change?',
                          subtitle=['Use the dropdowns to switch target and model']))
visual10

alt.VConcatChart(...)

> **Key takeaways**
>
> - **Accuracy panel:** every model's test and CV R² is close to 0. Exposure and skill can't predict *which* jobs grew or shrank, even though they predict *pay levels* well (R² ≈ 0.42).
> - **Coefficient panel:** the only exposure-related interval that stays off zero is **AIOE for employment** (lower-skill decline). Everything else crosses zero.
> - **Importance panel:** for wage growth the 2022 wage dominates and exposure barely matters. For employment growth, **AI exposure is one of the most important inputs** (the top input for both tree models), alongside skill tier and starting size. Treat this as a hint, since overall accuracy is low.
> - **Honest summary:** there is a real but small early signal. AI exposure is not (yet) a major driver of job change.

---

---
# Part C — Conclusions

In [17]:
def effect_sentence(res, label):
    b, p = res.params, res.pvalues
    lo_eff, hi_eff = 100 * b['AIOE'], 100 * (b['AIOE'] + b['AIOE:high_skill'])
    lo_sig = 'statistically significant' if p['AIOE'] < 0.05 else 'not statistically significant'
    int_sig = ('the difference between tiers IS statistically significant' if p['AIOE:high_skill'] < 0.05
               else 'the difference between tiers is NOT statistically significant')
    return (f'- {label}: +1 SD of AI exposure → {lo_eff:+.1f} pts for lower-skill jobs ({lo_sig}, p = {p["AIOE"]:.3f}) '
            f'and {hi_eff:+.1f} pts for higher-skill jobs; {int_sig} (p = {p["AIOE:high_skill"]:.3f}).')

best = chg_metrics.sort_values('cv_r2_mean', ascending=False).groupby('target').head(1)
lvl_cv = level_metrics.set_index('model')['cv_r2_mean']

print('WHAT WE FOUND\n')
print(f'1. Levels (2025): exposure is highest in well-paid, higher-skill jobs. The OLS model explains '
      f'about {100*lvl_cv["OLS (with interaction)"]:.0f}% of the variation in pay (5-fold CV), and the complex models '
      f'do not beat it.')
print(f'\n2. Change (2022 → 2025), {len(chg)} occupations. Typical occupation: wages '
      f"{chg['wage_growth_pct'].median():+.1f}%, employment {chg['emp_growth_pct'].median():+.1f}%.")
for t, res in ols_change.items():
    print(effect_sentence(res, TARGET_NAMES[t]))
print('\n3. How predictable is change?')
for _, r in best.iterrows():
    print(f"- {r['target']}: best model is {r['model']} with 5-fold CV R² = {r['cv_r2_mean']:.3f}. "
          + ('Exposure and skill explain very little of who grew or shrank.' if r['cv_r2_mean'] < 0.1
             else 'The inputs explain a meaningful share of who grew or shrank.'))

w, e = ols_change['wage_growth'], ols_change['emp_growth']
neg = [TARGET_NAMES[t] for t, r in ols_change.items()
       if (r.params['AIOE'] < 0 and r.pvalues['AIOE'] < 0.05)
       or (r.params['AIOE'] + r.params['AIOE:high_skill'] < 0 and r.pvalues['AIOE:high_skill'] < 0.05)]
print('\nBOTTOM LINE')
if neg:
    print(f'- We find signs that more-exposed jobs fell behind on: {", ".join(neg)}. '
          'That is consistent with early AI displacement, but 3 years is short, so treat it as early evidence.')
else:
    print('- We find no clear sign that more-exposed jobs lost pay or workers relative to other jobs between 2022 and 2025. '
          'So far, exposure looks like it tracks who AI touches, not who it replaces.')

WHAT WE FOUND

1. Levels (2025): exposure is highest in well-paid, higher-skill jobs. The OLS model explains about 42% of the variation in pay (5-fold CV), and the complex models do not beat it.

2. Change (2022 → 2025), 661 occupations. Typical occupation: wages +13.1%, employment +2.9%.
- Wage growth: +1 SD of AI exposure → +0.1 pts for lower-skill jobs (not statistically significant, p = 0.618) and -0.1 pts for higher-skill jobs; the difference between tiers is NOT statistically significant (p = 0.838).
- Employment growth: +1 SD of AI exposure → -2.8 pts for lower-skill jobs (statistically significant, p = 0.021) and +0.6 pts for higher-skill jobs; the difference between tiers is NOT statistically significant (p = 0.114).

3. How predictable is change?
- Wage growth: best model is Random Forest with 5-fold CV R² = 0.084. Exposure and skill explain very little of who grew or shrank.
- Employment growth: best model is OLS (with interaction) with 5-fold CV R² = 0.055. Exposure and ski

### Answering the research question
1. **Who is exposed?** AI exposure is concentrated in **well-paid, higher-skill jobs** such as tech, legal, finance, and management. It is not concentrated in the low-wage work people usually worry about.
2. **Is displacement showing up?** Not in pay: exposed jobs got about the same raises as everyone else from 2022 to 2025. In employment, there are **early signs among lower-skill exposed jobs**, mostly office/admin and sales, which shrank while the typical job grew. Higher-skill exposed jobs grew.
3. **High-skill vs low-skill:** the decline shows up only among lower-skill exposed jobs (p ≈ 0.02). Higher-skill exposed jobs grew, but the gap between tiers isn't statistically significant yet (p ≈ 0.11). The direction matches research from other countries, where AI-related displacement falls hardest on lower-skill workers (Ly et al., 2026).
4. **Predictability:** exposure and skill explain a lot of who is *paid* more (CV R² ≈ 0.42), but very little of who *grew or shrank* (CV R² near 0). Change is driven mostly by other forces.

**Bottom line:** exposure is highest in the jobs many of us are training for. So far, it looks like AI is reshaping lower-skill exposed work first rather than replacing high-skill workers.

## Limitations to mention in the presentation
- **Three years is short.** Effects of generative AI may take longer to show up in wages and headcounts.
- **2022 is a smoothed baseline.** BLS builds each May estimate from 3 years of survey panels (2019–2022 for May 2022), and response rates were lower during COVID.
- **Inflation and the post-COVID labor market** pushed wages up everywhere, especially in low-paid jobs. We handle this by comparing occupations against each other and controlling for the 2022 wage, but other shocks (remote work, interest rates, tech layoffs in 2023) happened at the same time as ChatGPT.
- **Correlation, not causation.** A difference in growth between exposed and unexposed jobs is consistent with AI effects but doesn't prove them.
- **Top-coded wages.** A few of the highest-paid occupations have no exact 2022 median (BLS `#`), so they drop out of the wage-growth analysis.
- **Skill tier is a proxy** based on SOC major group, not individual education requirements.